# Taller: Diseño de Bloques Completamente Aleatorizados (DBCA)

**Curso:** Análisis de Datos  
**Tema:** Sección 3 – Diseño de Bloques Completamente Aleatorizados

***

## Contexto del problema

Una empresa de comercio electrónico desea evaluar el efecto de **cuatro estrategias de marketing digital** sobre las **ventas diarias (en millones de pesos)** en su plataforma. Las estrategias evaluadas son:

| Tratamiento | Descripción |
|---|---|
| **Email Marketing** | Campaña por correo electrónico segmentada |
| **Publicidad Redes** | Anuncios pagados en redes sociales |
| **Influencers** | Colaboraciones con creadores de contenido |
| **Descuentos** | Promociones y cupones de descuento |

La empresa opera en **cuatro ciudades colombianas**, que actúan como **bloques** ya que el volumen de ventas varía naturalmente entre ciudades (Bogotá, Medellín, Cali y Barranquilla). En cada ciudad se aplicaron las cuatro estrategias con **8 réplicas** cada una.

### Diseño experimental

| Factor | Descripción |
|---|---|
| **Tratamiento** | Estrategia de marketing (4 niveles) |
| **Bloque** | Ciudad (4 bloques) |
| **Respuesta** | Ventas diarias en millones de pesos |
| **Réplicas** | 8 observaciones por celda |

> **Objetivo:** Determinar si alguna estrategia de marketing genera diferencias significativas en ventas, controlando la variabilidad entre ciudades mediante un DBCA.

***

## Contenido

1. Exploración inicial de los datos  
2. Visualización: perfiles de respuesta y distribuciones  
3. Modelo ANOVA – DBCA  
4. Eficiencia del bloqueo  
5. Comparaciones múltiples (Tukey HSD)  
6. Diagnóstico de supuestos  
7. Conclusiones

## Librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13
sns.set_theme(style='whitegrid', palette='Set2')
print("Librerías cargadas correctamente ✓")

***
# 1. Exploración Inicial de los Datos

Cargamos el archivo `diseno_bloques_aleatorizados.csv`, que contiene las ventas diarias registradas en cada ciudad para cada estrategia de marketing.

In [ ]:
# ------------------------------------------------------------------
# Carga de datos
# ------------------------------------------------------------------
df = pd.read_csv('diseno_bloques_aleatorizados.csv')

print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nTratamientos: {sorted(df['Tratamiento'].unique())}")
print(f"Bloques     : {sorted(df['Ciudad_Bloque'].unique())}")
print(f"\nObservaciones por celda (bloque × tratamiento):")
print(df.groupby(['Ciudad_Bloque', 'Tratamiento']).size().unstack().to_string())
print(f"\nTotal de observaciones por tratamiento: {df['Tratamiento'].value_counts().to_dict()}")

In [ ]:
# ------------------------------------------------------------------
# Estadísticas descriptivas
# ------------------------------------------------------------------
print("Estadísticas por TRATAMIENTO:")
print(df.groupby('Tratamiento')['Ventas'].describe().round(2).to_string())

print("\nEstadísticas por BLOQUE (ciudad):")
print(df.groupby('Ciudad_Bloque')['Ventas'].describe().round(2).to_string())

print("\nMedia general de ventas: {:.2f} M$".format(df['Ventas'].mean()))

# Tabla de medias en formato matricial (bloques × tratamientos)
print("\nTabla de medias – Ventas (M$):")
pivot = df.pivot_table(values='Ventas', index='Ciudad_Bloque',
                       columns='Tratamiento', aggfunc='mean').round(2)
pivot['Media_bloque'] = pivot.mean(axis=1).round(2)
pivot.loc['Media_trat'] = pivot.mean(axis=0).round(2)
print(pivot.to_string())

***
# 2. Visualización

## 2.1 Motivación del bloqueo

Antes de ajustar el modelo, es importante verificar visualmente si:
1. Las ciudades (bloques) producen **niveles distintos de ventas** — lo que justifica el bloqueo.
2. Los **perfiles de respuesta son aproximadamente paralelos** entre bloques — lo que valida el supuesto de *no interacción* del DBCA.

> Si los perfiles son paralelos, el efecto del tratamiento es **consistente** en todas las ciudades y el modelo aditivo $y_{ij} = \mu + \tau_i + \beta_j + \varepsilon_{ij}$ es apropiado.

In [ ]:
# ------------------------------------------------------------------
# Gráfico de perfiles (interaction plot) y distribuciones
# ------------------------------------------------------------------
orden_trat = ['Email Marketing', 'Publicidad Redes', 'Descuentos', 'Influencers']
colores_ciudades = sns.color_palette('Set1', 4)
ciudades = sorted(df['Ciudad_Bloque'].unique())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Panel 1: Perfiles de ventas por ciudad ---
medias_celda = df.groupby(['Ciudad_Bloque', 'Tratamiento'])['Ventas'].mean().reset_index()

for j, ciudad in enumerate(ciudades):
    sub = medias_celda[medias_celda['Ciudad_Bloque'] == ciudad]
    sub = sub.set_index('Tratamiento').reindex(orden_trat).reset_index()
    axes[0].plot(sub['Tratamiento'], sub['Ventas'],
                 'o-', label=ciudad, color=colores_ciudades[j],
                 linewidth=2, markersize=8)

axes[0].set_title('Perfiles de ventas por ciudad\n(líneas paralelas → aditividad del DBCA)')
axes[0].set_xlabel('Estrategia de marketing')
axes[0].set_ylabel('Ventas medias (M$)')
axes[0].legend(title='Ciudad', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)

# --- Panel 2: Boxplot por tratamiento ---
orden_box = df.groupby('Tratamiento')['Ventas'].median().sort_values().index.tolist()
sns.boxplot(data=df, x='Tratamiento', y='Ventas', order=orden_box, ax=axes[1], palette='Set2')
sns.stripplot(data=df, x='Tratamiento', y='Ventas', order=orden_box, ax=axes[1],
              color='black', alpha=0.5, jitter=True, size=4)
axes[1].set_title('Distribución de ventas por estrategia\n(todos los bloques combinados)')
axes[1].set_xlabel('Estrategia de marketing')
axes[1].set_ylabel('Ventas (M$)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print("Observaciones:")
print("  • Las líneas de perfil son aproximadamente paralelas → supuesto de aditividad razonable.")
print("  • Las ciudades muestran niveles distintos de ventas → el bloqueo por ciudad es pertinente.")

***
# 3. Modelo ANOVA – DBCA

## 3.1 Modelo estadístico

El DBCA asume el siguiente modelo lineal aditivo:

$$y_{ij} = \mu + \tau_i + \beta_j + \varepsilon_{ij}, \quad \varepsilon_{ij} \sim N(0, \sigma^2)$$

donde:
- $\mu$ = media general de ventas
- $\tau_i$ = efecto de la estrategia de marketing $i$ ($i = 1,\ldots,4$)
- $\beta_j$ = efecto de la ciudad $j$ ($j = 1,\ldots,4$)
- $\varepsilon_{ij}$ = error aleatorio

## 3.2 Hipótesis

Las hipótesis de interés son:

**Para tratamientos (estrategias):**
$$H_0^{(T)}: \tau_1 = \tau_2 = \tau_3 = \tau_4 = 0 \quad \text{vs} \quad H_1^{(T)}: \text{al menos un } \tau_i \neq 0$$

**Para bloques (ciudades):**
$$H_0^{(B)}: \beta_1 = \beta_2 = \beta_3 = \beta_4 = 0 \quad \text{vs} \quad H_1^{(B)}: \text{al menos un } \beta_j \neq 0$$

## 3.3 Descomposición de la variabilidad

$$SS_{Total} = SS_{Tratamientos} + SS_{Bloques} + SS_{Error}$$

| Fuente | GL | SS | MS | F |
|---|---|---|---|---|
| Tratamientos | $k-1 = 3$ | $SS_T$ | $MS_T$ | $F_T = MS_T/MS_E$ |
| Bloques | $b-1 = 3$ | $SS_B$ | $MS_B$ | $F_B = MS_B/MS_E$ |
| Error | $(k-1)(b-1) \times r = 9 \times 8 = 112$* | $SS_E$ | $MS_E$ | |
| Total | $kbr-1 = 127$ | $SS_{Tot}$ | | |

*Con $r=8$ réplicas por celda, los grados de libertad del error son $(kb-1)r - (k-1) - (b-1) = N - k - b + 1$.

In [ ]:
# ------------------------------------------------------------------
# Ajuste del modelo DBCA y tabla ANOVA
# ------------------------------------------------------------------
modelo_dbca = ols('Ventas ~ C(Tratamiento) + C(Ciudad_Bloque)', data=df).fit()
tabla_dbca  = anova_lm(modelo_dbca, typ=1)
tabla_dbca.index = ['Tratamiento', 'Bloque (Ciudad)', 'Error (Residuo)']
tabla_dbca.columns = ['GL', 'SS', 'MS', 'F', 'p-valor']

print("=" * 70)
print("    TABLA ANOVA – Diseño de Bloques Completamente Aleatorizado (DBCA)")
print("=" * 70)
print(tabla_dbca.round(4).to_string())
print("=" * 70)

p_trat   = tabla_dbca.loc['Tratamiento',       'p-valor']
p_bloque = tabla_dbca.loc['Bloque (Ciudad)',    'p-valor']
F_trat   = tabla_dbca.loc['Tratamiento',        'F']
F_bloque = tabla_dbca.loc['Bloque (Ciudad)',     'F']
R2_dbca  = modelo_dbca.rsquared

print(f"\nTratamiento    : F = {F_trat:.4f},  p = {p_trat:.4f}",
      "→ SIGNIFICATIVO ✓" if p_trat < 0.05 else "→ No significativo")
print(f"Bloque (Ciudad): F = {F_bloque:.4f},  p = {p_bloque:.4f}",
      "→ SIGNIFICATIVO (bloqueo útil) ✓" if p_bloque < 0.05 else "→ No significativo (bloqueo no necesario)")
print(f"\nR² del modelo DBCA: {R2_dbca:.4f}")
print(f"  → El modelo explica el {R2_dbca*100:.1f}% de la variabilidad total en ventas.")

***
# 4. Eficiencia del Bloqueo

## ¿Valió la pena bloquear por ciudad?

Una forma objetiva de responder esta pregunta es comparar el $MS_E$ del **DBCA** con el $MS_E$ que se hubiera obtenido si se hubiera ignorado el bloque y analizado los datos como un **DCA**.

La **Eficiencia Relativa (ER)** del DBCA respecto al DCA es:

$$ER = \frac{MS_{E,\text{DCA}}}{MS_{E,\text{DBCA}}}$$

- $ER > 1$: el DBCA es más preciso — el bloqueo fue útil.
- $ER = 1$: el bloqueo no aportó nada.
- $ER < 1$: el bloqueo perjudicó (caso raro; señala que el factor de bloqueo era irrelevante).

> Un $ER = 2.5$ significa que el DCA necesitaría **2.5 veces más réplicas** por tratamiento para lograr la misma precisión que el DBCA.

In [ ]:
# ------------------------------------------------------------------
# Comparación DCA vs DBCA: MS_E y Eficiencia Relativa
# ------------------------------------------------------------------
modelo_dca = ols('Ventas ~ C(Tratamiento)', data=df).fit()
tabla_dca  = anova_lm(modelo_dca, typ=1)

MS_E_dca  = tabla_dca.iloc[1]['mean_sq']
MS_E_dbca = tabla_dbca.loc['Error (Residuo)', 'MS']
ER = MS_E_dca / MS_E_dbca

print("Comparación de varianza residual: DCA vs DBCA\n")
print(f"  MS_E  (DCA  – bloques ignorados) : {MS_E_dca:.4f}")
print(f"  MS_E  (DBCA – bloques incluidos) : {MS_E_dbca:.4f}")
print(f"\n  Eficiencia Relativa (ER)         : {ER:.2f}")
print()

if ER > 1.05:
    print(f"  → El DBCA es {ER:.2f}x más eficiente que el DCA.")
    print(f"     El DCA necesitaría ≈ {int(np.ceil(8 * ER))} réplicas por tratamiento")
    print(f"     para alcanzar la misma precisión que el DBCA con 8 réplicas.")
    print(f"\n  → Conclusión: bloquear por ciudad fue una decisión acertada.")
elif ER < 0.95:
    print("  → El bloqueo redujo la eficiencia. La variabilidad entre ciudades")
    print("     no era suficientemente grande para justificar el bloqueo.")
else:
    print("  → El bloqueo no aportó ventaja apreciable en eficiencia.")

# Visualización comparativa de MS_E
fig, ax = plt.subplots(figsize=(7, 4))
diseños = ['DCA\n(sin bloques)', 'DBCA\n(con bloques)']
ms_vals = [MS_E_dca, MS_E_dbca]
colores  = ['#e07b7b', '#6abf69']
bars = ax.bar(diseños, ms_vals, color=colores, edgecolor='black', width=0.4)
ax.set_ylabel('Varianza residual (MS_E)')
ax.set_title(f'Reducción del error al bloquear\nER = {ER:.2f}')
for bar, val in zip(bars, ms_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

***
# 5. Comparaciones Múltiples (Tukey HSD)

Si el ANOVA rechaza $H_0^{(T)}$, es necesario identificar **qué pares de tratamientos difieren** entre sí. El método de **Tukey HSD** controla la tasa de error familiar (FWER) al $\alpha = 0.05$ comparando simultáneamente todos los pares posibles.

> **Nota:** Las comparaciones se hacen sobre los **tratamientos**, no sobre los bloques. Los bloques son un factor de control, no de interés primario.

In [ ]:
# ------------------------------------------------------------------
# Comparaciones múltiples post-DBCA (Tukey HSD)
# ------------------------------------------------------------------
mc_dbca = pairwise_tukeyhsd(df['Ventas'], df['Tratamiento'], alpha=0.05)

print("=" * 70)
print("   COMPARACIONES MÚLTIPLES – Tukey HSD  (α = 0.05)  |  DBCA")
print("=" * 70)
print(mc_dbca.summary())

print("\nInterpretación:")
print("  'reject=True'  → diferencia estadísticamente significativa.")
print("  'reject=False' → no hay evidencia suficiente de diferencia.")

In [ ]:
# ------------------------------------------------------------------
# Visualización de comparaciones múltiples
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Intervalos simultáneos de Tukey
mc_dbca.plot_simultaneous(ax=axes[0], ylabel='Estrategia', xlabel='Ventas (M$)')
axes[0].set_title('Intervalos simultáneos Tukey HSD\n(α = 0.05)')

# Panel 2: Medias por tratamiento ± error estándar (del DBCA)
medias_trat = df.groupby('Tratamiento')['Ventas'].mean().sort_values()
sem_trat    = df.groupby('Tratamiento')['Ventas'].sem().reindex(medias_trat.index)
colores_bars = sns.color_palette('Set2', len(medias_trat))

axes[1].barh(medias_trat.index, medias_trat.values,
             xerr=sem_trat.values, capsize=5,
             color=colores_bars, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Ventas promedio (M$)')
axes[1].set_title('Medias por estrategia ± error estándar')
axes[1].axvline(df['Ventas'].mean(), color='red', linestyle='--', linewidth=1.2,
                label=f'Media general: {df["Ventas"].mean():.1f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

***
# 6. Diagnóstico de Supuestos del DBCA

El modelo DBCA asume que los errores $\varepsilon_{ij}$ son:

1. **Independientes** entre sí.
2. **Homocedásticos** — varianza constante $\sigma^2$ en todos los grupos.
3. **Normalmente distribuidos** — $\varepsilon_{ij} \sim N(0, \sigma^2)$.

Verificamos estos supuestos a través de gráficos de diagnóstico y pruebas formales.

In [ ]:
# ------------------------------------------------------------------
# Diagnóstico gráfico de supuestos del DBCA
# ------------------------------------------------------------------
df['Residuos']  = modelo_dbca.resid
df['Ajustados'] = modelo_dbca.fittedvalues

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Q-Q plot — normalidad
sm.qqplot(df['Residuos'], line='s', ax=axes[0], alpha=0.7, color='steelblue')
axes[0].set_title('Q-Q Plot\nNormalidad de residuos')

# Residuos vs. valores ajustados — homocedasticidad
axes[1].scatter(df['Ajustados'], df['Residuos'],
                color='coral', edgecolors='black', alpha=0.6, s=30)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_title('Residuos vs. Ajustados\nHomocedasticidad')
axes[1].set_xlabel('Valor ajustado')
axes[1].set_ylabel('Residuo')

# Residuos por tratamiento
df.boxplot(column='Residuos', by='Tratamiento', ax=axes[2], grid=False)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_title('Residuos por tratamiento\n(deben ser homogéneos)')
axes[2].set_xlabel('Tratamiento')
axes[2].set_ylabel('Residuo')
plt.sca(axes[2])
plt.xticks(rotation=20, ha='right', fontsize=8)
plt.suptitle('')

# Residuos por bloque — verificar que el bloqueo aisló la variabilidad
df.boxplot(column='Residuos', by='Ciudad_Bloque', ax=axes[3], grid=False)
axes[3].axhline(0, color='red', linestyle='--')
axes[3].set_title('Residuos por bloque\n(variabilidad entre ciudades aislada)')
axes[3].set_xlabel('Ciudad')
axes[3].set_ylabel('Residuo')
plt.sca(axes[3])
plt.xticks(rotation=20, ha='right', fontsize=8)
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Pruebas formales de supuestos
# ------------------------------------------------------------------

# 1. Normalidad: Shapiro-Wilk sobre los residuos
stat_sw, p_sw = stats.shapiro(df['Residuos'])

# 2. Homocedasticidad: Levene (robusto a no-normalidad)
grupos = [grp['Residuos'].values for _, grp in df.groupby('Tratamiento')]
stat_lev, p_lev = stats.levene(*grupos, center='median')

# 3. Homocedasticidad: Bartlett (asume normalidad)
stat_bart, p_bart = stats.bartlett(*grupos)

print("=" * 60)
print("   PRUEBAS FORMALES DE SUPUESTOS DEL DBCA")
print("=" * 60)
print(f"\n1. Normalidad (Shapiro-Wilk):")
print(f"   W = {stat_sw:.4f},  p = {p_sw:.4f}",
      "→ Normal ✓" if p_sw > 0.05 else "→ ⚠  Posible desviación de normalidad")

print(f"\n2. Homocedasticidad (Levene):")
print(f"   W = {stat_lev:.4f},  p = {p_lev:.4f}",
      "→ Varianzas iguales ✓" if p_lev > 0.05 else "→ ⚠  Varianzas posiblemente distintas")

print(f"\n3. Homocedasticidad (Bartlett):")
print(f"   χ² = {stat_bart:.4f},  p = {p_bart:.4f}",
      "→ Varianzas iguales ✓" if p_bart > 0.05 else "→ ⚠  Revisar homocedasticidad")

print("\n" + "=" * 60)
print("Nota: Con n=128, Shapiro-Wilk puede ser sensible a desviaciones")
print("menores. Revisar el Q-Q plot para una evaluación visual integral.")

***
# 7. Conclusiones

## Resumen del análisis DBCA

| Sección | Concepto | Resultado |
|---|---|---|
| **Exploración** | Tabla de medias (bloque × trat.) | 4 bloques × 4 tratamientos × 8 réplicas |
| **Visualización** | Perfiles de respuesta (interaction plot) | Líneas aproximadamente paralelas → aditividad ✓ |
| **ANOVA DBCA** | $F_T$, $p$-valor tratamiento | Interpretar con los resultados obtenidos |
| **ANOVA DBCA** | $F_B$, $p$-valor bloque | Bloqueo significativo → reducción del error ✓ |
| **Eficiencia** | $ER = MS_{E,DCA}/MS_{E,DBCA}$ | Cuantifica ganancia del bloqueo |
| **Comparaciones** | Tukey HSD | Identifica pares de estrategias distintas |
| **Supuestos** | Shapiro-Wilk, Levene | Validan normalidad y homocedasticidad |

## Flujo del análisis

```
1. Exploración descriptiva (medias, variabilidad)
         ↓
2. Verificar perfiles de respuesta (aditividad del DBCA)
         ↓
3. Ajustar modelo: Ventas ~ C(Tratamiento) + C(Ciudad_Bloque)
         ↓
4. Tabla ANOVA → ¿es significativo el tratamiento?
         ↓  (sí)
5. Eficiencia Relativa → ¿valió la pena bloquear?
         ↓
6. Tukey HSD → ¿qué estrategias difieren entre sí?
         ↓
7. Diagnóstico de residuos → validar supuestos
         ↓
8. Conclusión práctica para la empresa
```

## Preguntas de reflexión

1. ¿Qué estrategia de marketing recomendaría a la empresa y por qué?
2. ¿Qué hubiera pasado si no se hubiera bloqueado por ciudad? ¿Habría cambiado la conclusión?
3. Si los perfiles no fueran paralelos, ¿qué modelo alternativo sería más apropiado?
4. ¿Qué haría si la prueba de Shapiro-Wilk rechazara la normalidad de los residuos?